In [ ]:
#==================================
# DEM 提取脚本 - 终极多线程防抖版
# 核心解决：ERROR 000725 (数据集已存在)
# 数据不可用，此脚本作废
# 数据不可用，此脚本作废
# 数据不可用，此脚本作废
# 数据不可用，此脚本作废
# 数据不可用，此脚本作废
# 数据不可用，此脚本作废
# 数据不可用，此脚本作废
# 数据不可用，此脚本作废
# 数据不可用，此脚本作废
#==================================\




import arcpy
import os
import time
import sys
from concurrent.futures import ThreadPoolExecutor
import threading

# ==================== 1. 参数设置 ====================
service = "https://elevation.nationalmap.gov/arcgis/services/3DEPElevation/ImageServer"
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\GenerateTessellation_1_ExportFeatures2"
out_folder = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\raw\DEM\DEM_Evelation_Fishnet"

USE_MULTITHREAD = True  
MAX_WORKERS = 4         # 保持4线程，稳字当头

# ==================== 2. 环境初始化 ====================
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

arcpy.env.overwriteOutput = True
arcpy.env.cellSize = 1 
arcpy.env.extent = None 

arcpy.management.SelectLayerByAttribute(buffer_layer, "CLEAR_SELECTION")
total_points = int(arcpy.management.GetCount(buffer_layer)[0])

success_count = 0
skip_count = 0
error_count = 0
lock = threading.Lock() 
start_time = time.time()

# ==================== 3. 核心提取函数 ====================
def download_dem_task(oid, wkt):
    global success_count, skip_count, error_count
    
    out_path = os.path.join(out_folder, f"Extract_DEM_{oid}.tif")
    if os.path.exists(out_path):
        with lock: skip_count += 1
        return

    # 为每个OID分配独立的图层名
    local_dem_layer = f"Raw_Elev_Layer_{oid}"
    feat_geom = arcpy.FromWKT(wkt)
    
    # 【核心修复】：安全的图层创建工厂
    # 无论是被上一次崩溃残留，还是重试引发的冲突，都在这里一并清扫
    def safe_make_layer():
        if arcpy.Exists(local_dem_layer):
            arcpy.management.Delete(local_dem_layer)
        arcpy.management.MakeImageServerLayer(service, local_dem_layer, processing_template="None")

    try:
        # 第一次尝试
        safe_make_layer()
        extracted = arcpy.sa.ExtractByMask(local_dem_layer, feat_geom)
        extracted.save(out_path)
        
        with lock:
            success_count += 1
            processed = success_count + skip_count
            if processed % 5 == 0 or processed == total_points:
                elapsed = time.time() - start_time
                speed = processed / elapsed
                remaining = (total_points - processed) / speed / 60
                print(f"进度: {processed}/{total_points} | 成功: {success_count} | 速度: {speed:.2f} 点/秒 | 预计剩余: {remaining:.1f} 分钟")

    except Exception as e:
        # 捕获网络异常，进入重试阶段
        time.sleep(2)
        try:
            # 再次调用安全创建函数，它会先清理掉 try 块里可能残留的损坏图层
            safe_make_layer()
            extracted = arcpy.sa.ExtractByMask(local_dem_layer, feat_geom)
            extracted.save(out_path)
            
            with lock:
                success_count += 1
                print(f"--- OID {oid} 首次断开，但重试成功！ ---")
        except Exception as retry_e:
            with lock:
                error_count += 1
                print(f"!!! OID {oid} 最终失败: {retry_e}")
                
    finally:
        # 兜底清理：执行完毕后无条件删除内存图层
        if arcpy.Exists(local_dem_layer):
            arcpy.management.Delete(local_dem_layer)

# ==================== 4. 任务执行 ====================
print(f"--- 任务启动 (多线程: {USE_MULTITHREAD}, 线程数: {MAX_WORKERS}) ---")

tasks = []
with arcpy.da.SearchCursor(buffer_layer, ["OID@", "SHAPE@WKT"]) as cursor:
    for row in cursor:
        tasks.append((row[0], row[1]))

if USE_MULTITHREAD:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for oid, wkt in tasks:
            executor.submit(download_dem_task, oid, wkt)
else:
    for oid, wkt in tasks:
        download_dem_task(oid, wkt)

# ==================== 5. 总结 ====================
arcpy.CheckInExtension("Spatial")
total_time = (time.time() - start_time) / 60
print(f"\n--- 任务结束 --- \n成功: {success_count} | 跳过: {skip_count} | 失败: {error_count} \n总耗时: {total_time:.2f} 分钟")

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000002A2916AA550>>
Traceback (most recent call last):
  File "c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\ipykernel\ipkernel.py", line 790, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\threading.py", line 1501, in enumerate
    def enumerate():
    
KeyboardInterrupt: 


--- 任务启动 (多线程: True, 线程数: 4) ---
